# UA-OSM — Validation Notebook
**Uncertainty-Aware Object-Shaped Merging to Reduce Object Loss in 360° Surround-View Stitching**
*Rohit Joshi, EIMT, Switzerland*

This notebook reproduces the validation results of the paper with the same code that generated them:

| Paper section | Notebook section |
|---|---|
| III Problem formulation (Eq. 1–6) | §3 Geometry, footprint model, analytic and pixel-level OVI |
| IV Feasibility of straight seams (Prop. 1–2, Table II, Figs. 2–3) | §4 |
| V UA-OSM method, VI baselines/ablations | §5 |
| Fig. 1 qualitative result | §6 |
| VII-A/B Main benchmark, statistics, density (Tables VI–VII, Fig. 5) | §7 |
| VII-C/D/E Planning-model mismatch, detection error, latency | §8 |
| VII Robustness, parameters, driver warnings | §9 |
| VI/VII Scenario catalog: time of day × weather × density | §10 |
| VII Unknown-object rule and calibration false alarms | §11 |
| VII Real-world appearance check (photographs) | §12 |
| Reference values from the paper, parameters, limitations | §13 |

**Run modes.** `QUICK=True` (default) uses small scene counts and finishes in roughly 10–20 minutes on a Colab CPU; numbers will be noisier than in the paper. `QUICK=False` uses the paper's scene counts (several hours on one CPU core). Every section prints the paper's reference values next to the recomputed ones.

**Honesty note.** All merging results are simulated (ray-traced four-camera model, flat-ground top view). The photographs in §12 test an appearance assumption only; they do not validate merging.

## 1. Setup

In [ ]:
!pip -q install opencv-python-headless scipy matplotlib >/dev/null 2>&1
QUICK = True        # set False to reproduce the paper's scene counts (slow)
import numpy as np, json, time, math, sys, os
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi':110,'font.size':9})
N = dict(main=10 if QUICK else 105, sweep=6 if QUICK else 30, robust=6 if QUICK else 20,
         catalog=3 if QUICK else 10, unknown=9 if QUICK else 30, calib=2 if QUICK else 8, grid=0.1 if QUICK else 0.05)
print('Scene counts:', N)
REF = json.loads(r'''{"main_pixel": {"B1 Fixed seam": [0.427, 63.0], "B2 Straight dynamic seam": [0.799, 19.9], "B3 One camera per object": [0.734, 27.6], "B4 Curved mask-avoiding seam": [0.774, 24.0], "A5 w/o joint assignment": [0.785, 22.4], "A6 w/o visibility": [0.896, 9.2], "UA-OSM (full)": [0.916, 7.0]}, "perk_lost": {"B2": [3.6, 6.9, 13.8, 29.3, 45.7], "UA-OSM": [0.0, 1.2, 3.8, 10.0, 20.0]}, "feasibility": {"15-75": 91.0, "10-80": 64.7, "5-85": 35.3, "0-90": 9.0, "bollard 15-75": 70.1}, "sigma": {"0 cm": [20.0, 6.3, 6.3], "20 cm": [24.7, 12.3, 9.8], "30 cm": [28.3, 19.8, 14.7]}, "latency": {"100 ms pred": [20.8, 6.5], "300 ms pred": [23.5, 7.3], "300 ms no pred": [25.2, 10.8]}, "catalog_B2": [[23.0, 22.5, 23.5, 25.0], [24.000000000000004, 23.0, 25.0, 25.5], [23.5, 23.5, 24.000000000000004, 27.0]], "catalog_UA": [[5.0, 5.500000000000001, 4.5, 8.5], [6.000000000000001, 4.5, 9.0, 12.000000000000002], [6.0, 4.5, 9.0, 13.0]], "catalog_L5": [[16.5, 5.5, 18.0, 26.0], [24.0, 22.0, 26.0, 37.5], [27.0, 24.0, 27.5, 40.0]], "unknown": {"disabled": [0.589, 50.0], "60": [0.687, 40.0], "45": [0.802, 23.3], "35": [0.804, 23.3]}, "photo_contrast": {"IMG_2091": {"light": "night, floodlight", "asphalt_rgb": [185.0, 168.0, 147.0], "shirt": {"rgb": [151.0, 85.0, 68.0], "dk": 65.3, "weber": -0.42}, "jeans": {"rgb": [119.0, 121.0, 123.0], "dk": 45.8, "weber": -0.29}, "shoes": {"rgb": [164.0, 152.0, 135.0], "dk": 16.4, "weber": -0.1}}, "IMG_2090": {"light": "evening, walking", "asphalt_rgb": [174.0, 168.0, 163.0], "shirt": {"rgb": [148.0, 83.0, 69.0], "dk": 68.4, "weber": -0.43}, "jeans": {"rgb": [130.0, 134.0, 141.0], "dk": 33.6, "weber": -0.21}}, "IMG_2089": {"light": "evening, floodlight", "asphalt_rgb": [165.0, 146.0, 126.0], "shirt": {"rgb": [132.0, 70.0, 57.0], "dk": 59.1, "weber": -0.44}, "jeans": {"rgb": [103.0, 105.0, 109.0], "dk": 40.2, "weber": -0.3}, "shoes": {"rgb": [111.0, 99.0, 86.0], "dk": 46.9, "weber": -0.32}}, "IMG_2088": {"light": "dusk, street light", "asphalt_rgb": [172.0, 149.0, 128.0], "shirt": {"rgb": [154.0, 76.0, 59.0], "dk": 53.8, "weber": -0.4}, "jeans": {"rgb": [117.0, 113.0, 113.0], "dk": 35.6, "weber": -0.25}, "shoes": {"rgb": [178.0, 161.0, 138.0], "dk": 9.3, "weber": 0.07}}, "IMG_2085": {"light": "night, floodlight", "asphalt_rgb": [119.0, 103.0, 88.0], "shirt": {"rgb": [191.0, 101.0, 80.0], "dk": 27.2, "weber": 0.13}, "jeans": {"rgb": [127.0, 127.0, 135.0], "dk": 26.1, "weber": 0.21}}, "IMG_2083": {"light": "dusk, floodlight", "asphalt_rgb": [125.0, 110.0, 90.0], "shirt": {"rgb": [180.0, 91.0, 70.0], "dk": 31.4, "weber": -0.03}, "jeans": {"rgb": [115.0, 119.0, 127.0], "dk": 18.7, "weber": 0.07}, "shoes": {"rgb": [158.0, 147.0, 132.0], "dk": 37.5, "weber": 0.33}}}}''')

## 2. Simulation library
The next cells write the simulator and experiment harness to disk exactly as used for the paper.

* `sim2.py` — camera model (equidistant fisheye, 192°), ray tracer, flat-ground top view, exact footprint model (Eq. 2), merge weights (Eq. 3), analytic OVI (Eq. 4–5), pixel-level OVI (Eq. 6), all merging methods (fixed, dynamic straight, curved mask-avoiding, one-camera-per-object, UA-OSM and ablations), escalation ladder, visibility set, unexplained-content rule (Eq. 11).
* `scen.py` — body colours of rendered pedestrians and children.
* `e2.py` — perception model: Ornstein–Uhlenbeck position error, latency with velocity prediction, track losses and coasting.
* `e3.py` — static scene rendering helpers.
* `e5.py`, `e6.py` — random corner scenes and the pixel-level benchmark harness.
* `e9.py` — robustness harness (parameters, perception faults, rear corner with ego motion, covariance miscalibration, warning logging).
* `cond.py` — scenario catalog (time of day × weather → perception degradation).

In [ ]:
%%writefile sim2.py
import numpy as np, cv2, math
HX, HY = 0.95, 2.3
M0=0.12; LAM_U=0.03; LAM_S=0.25; L2_THR=0.95; L5_THR=0.9
BR=7.2; BN=440
CURVE_R=np.arange(0.05,9.0,0.1); CURVE_A=np.arange(0.0,90.01,2.5)
DEG = 180/math.pi
CAM_IDS = ['F','B','L','R']
SEAM_MIN, SEAM_MAX = 15, 75
CAR_H = 1.45

def norm(v): return v/np.linalg.norm(v)

class Cam:
    def __init__(s, pos, fwd):
        s.pos = np.array(pos, float); f = norm(np.array(fwd, float)); up = np.array([0,0,1.0])
        d = norm(-up - np.dot(-up, f)*f); r = np.cross(f, d)
        s.R = np.stack([r, d, f])        # rows: cam axes in world
        s.h = s.pos[2]

def pitched(yaw_vec, pitch_deg):
    y = norm(np.array([yaw_vec[0], yaw_vec[1], 0.0])); p = math.radians(pitch_deg)
    return np.array([y[0]*math.cos(p), y[1]*math.cos(p), -math.sin(p)])

CAMS = {'F': Cam((0,-2.35,0.65), pitched((0,-1),30)), 'B': Cam((0,2.35,0.95), pitched((0,1),35)),
        'L': Cam((-1.0,-0.9,1.0), pitched((-1,0),45)), 'R': Cam((1.0,-0.9,1.0), pitched((1,0),45))}

# ---------------- analytic model (same as web demo) ----------------
def sees(c,x,y):
    return (y<=-HY) if c=='F' else (y>=HY) if c=='B' else (x<=-HX) if c=='L' else (x>=HX)
CORNERS=[dict(a='F',b='L',cx=-HX,cy=-HY,sx=-1,sy=-1),dict(a='F',b='R',cx=HX,cy=-HY,sx=1,sy=-1),
         dict(a='B',b='L',cx=-HX,cy=HY,sx=-1,sy=1),dict(a='B',b='R',cx=HX,cy=HY,sx=1,sy=1)]
def corner_of(x,y):
    ya=-1 if y<=-HY else (1 if y>=HY else 0); xa=-1 if x<=-HX else (1 if x>=HX else 0)
    if not ya or not xa: return -1
    return (0 if xa<0 else 1) if ya<0 else (2 if xa<0 else 3)
def sstep(e0,e1,x):
    t=min(1.0,max(0.0,(x-e0)/(e1-e0))); return t*t*(3-2*t)
def cor_inside(c,x,y):
    px,py=x-c['x1'],y-c['y1']; s=px*c['dx']+py*c['dy']; p=abs(px*c['dy']-py*c['dx'])
    if s<-c['rad']: return False
    if s<0: return math.hypot(px,py)<=c['rad']
    if s>c['len']: return False
    return p<=c['rad']+c['slope']*s
CAM_CODE={'F':0,'B':1,'L':2,'R':3}
def weights(m,x,y):
    out={'F':0.0,'B':0.0,'L':0.0,'R':0.0}
    bg=m.get('blobgrid')
    if bg is not None:
        i=int((x+BR)/(2*BR)*BN); j=int((y+BR)/(2*BR)*BN)
        if 0<=i<BN and 0<=j<BN and bg[j,i]>=0:
            c=CAM_IDS[bg[j,i]]
            if sees(c,x,y): out[c]=1.0; return out
    for c in m['corridors']:
        if sees(c['owner'],x,y) and cor_inside(c,x,y): out[c['owner']]=1.0; return out
    k=corner_of(x,y)
    if k<0:
        if y<=-HY: out['F']=1
        elif y>=HY: out['B']=1
        elif x<=-HX: out['L']=1
        elif x>=HX: out['R']=1
        return out
    C=CORNERS[k]; u=(x-C['cx'])*C['sx']; v=(y-C['cy'])*C['sy']; a=math.atan2(u,v)*DEG
    phi=m['seams'][k]
    cv=m.get('curve')
    if cv is not None and cv[k] is not None:
        phi=float(np.interp(math.hypot(u,v),CURVE_R,cv[k]))
    b=m['band']; wa=1-sstep(phi-b,phi+b,a); out[C['a']]=wa; out[C['b']]=1-wa; return out
STREAK_CAP=7.0
LMAX=10.5   # beyond the display corner (display half-width 7.2 m)
def streak(c,o):
    C=CAMS[c]; dx=o['x']-C.pos[0]; dy=o['y']-C.pos[1]; d=math.hypot(dx,dy); dx/=d; dy/=d
    # exact flat-ground projection of the object top; objects at/above camera height reach the display edge
    L=LMAX if o['h']>=C.h*0.999 else min(d*o['h']/(C.h-o['h']),LMAX)
    return dict(x1=o['x'],y1=o['y'],x2=o['x']+dx*L,y2=o['y']+dy*L,dx=dx,dy=dy,len=L,d=d)
NS=18
def eval_obj(m,o):
    lst=[]
    for c in CAM_IDS:
        if not sees(c,o['x'],o['y']): continue
        s=streak(c,o); acc=0; cnt=0
        for i in range(NS):
            t=(i+0.5)/NS; px=s['x1']+(s['x2']-s['x1'])*t; py=s['y1']+(s['y2']-s['y1'])*t
            if abs(px)>BR or abs(py)>BR: continue   # only what is on screen counts
            cnt+=1
            if sees(c,px,py): acc+=weights(m,px,py)[c]
        lst.append((acc/max(cnt,1),c))
    lst.sort(reverse=True)
    best=lst[0][0] if lst else 0; second=lst[1][0] if len(lst)>1 else 0
    return dict(best=best,second=second,score=max(0,min(1,best-0.5*second)),owner=lst[0][1] if lst else None)
def status(r):
    if r['best']<0.55: return 'LOST'
    if r['second']>0.3: return 'DOUBLED'
    if r['best']<0.9: return 'CUT'
    return 'VISIBLE'
def relevant(o):
    s=set()
    for c in CAM_IDS:
        if not sees(c,o['x'],o['y']): continue
        st=streak(c,o)
        for i in range(7):
            t=i/6; k=corner_of(st['x1']+(st['x2']-st['x1'])*t, st['y1']+(st['y2']-st['y1'])*t)
            if k>=0: s.add(k)
    return s
def make_method(kind,band=6,sigma=0.0):
    return dict(kind=kind,band=band,sigma=sigma,seams=[45.0]*4,corridors=[],owners={},motion=0.0)
def update_method(m,est,dt):
    if m['kind']=='dynamic':
        for k in range(4):
            rel=[o for o in est if k in relevant(o)]; prev=m['seams'][k]; target=45.0
            if rel:
                bc=1e9; phi=SEAM_MIN
                while phi<=SEAM_MAX+1e-6:
                    m['seams'][k]=phi; cost=sum(1-eval_obj(m,o)['score'] for o in rel)+0.002*abs(phi-prev)+0.001*abs(phi-45)
                    if cost<bc: bc=cost; target=phi
                    phi+=2.5
            step=max(-90*dt,min(90*dt,target-prev)); m['seams'][k]=prev+step
    elif m['kind']=='shape':
        cor=[]
        for o in est:
            cams=[c for c in CAM_IDS if sees(c,o['x'],o['y'])]
            if not cams: continue
            st={c:streak(c,o) for c in cams}; owner=min(cams,key=lambda c:st[c]['d'])
            prev=m['owners'].get(o['id'])
            if prev in cams and prev!=owner and st[prev]['d']<st[owner]['d']*1.15: owner=prev
            m['owners'][o['id']]=owner
            for c in cams:
                s=st[c]; cor.append(dict(owner=owner,x1=s['x1'],y1=s['y1'],dx=s['dx'],dy=s['dy'],len=s['len'],
                    rad=o['r']+0.12+2*m['sigma'],slope=o['r']/s['d']+0.02))
        m['corridors']=cor

# ---------------- ground texture ----------------
TR=12.0; TN=1200
def make_ground():
    rng=np.random.default_rng(3)
    base=np.full((TN,TN,3),(92,96,99),np.float32)
    n=rng.normal(0,1,(TN//4,TN//4)).astype(np.float32); n=cv2.resize(n,(TN,TN),interpolation=cv2.INTER_CUBIC)
    base+=n[...,None]*5+rng.normal(0,4,(TN,TN,1)).astype(np.float32)
    P=lambda x,y:(int((x+TR)/(2*TR)*TN),int((y+TR)/(2*TR)*TN))
    img=np.clip(base,0,255).astype(np.uint8)
    W=(222,222,215); lw=max(2,int(0.12/(2*TR)*TN))
    for x in (-1.45,1.45,4.2,-4.2):   # parking stall side lines
        cv2.line(img,P(x,-2.9),P(x,3.2),W,lw)
    for xs in ((1.45,4.2),(-4.2,-1.45)): cv2.line(img,P(xs[0],3.2),P(xs[1],3.2),W,lw)
    cv2.line(img,P(-1.45,3.2),P(1.45,3.2),W,lw)
    for y0 in np.arange(-11,11,2.0):  # dashed aisle line
        cv2.line(img,P(-7.5,y0),P(-7.5,y0+1.0),W,lw); cv2.line(img,P(7.5,y0),P(7.5,y0+1.0),W,lw)
    for i in range(7):  # zebra in front
        x=-3.0+i*1.0; cv2.rectangle(img,P(x,-6.6),P(x+0.5,-5.4),(210,210,200),-1)
    cv2.circle(img,P(3.2,-4.6),int(0.35/(2*TR)*TN),(70,72,74),-1)
    return img
GROUND=make_ground()
def ground_color(gx,gy):
    gx=gx.ravel(); gy=gy.ravel(); n=gx.size; W=2048; rows=(n+W-1)//W; pad=rows*W-n
    u=np.pad(((gx+TR)/(2*TR)*TN).astype(np.float32),(0,pad)).reshape(rows,W)
    v=np.pad(((gy+TR)/(2*TR)*TN).astype(np.float32),(0,pad)).reshape(rows,W)
    out=cv2.remap(GROUND,u,v,cv2.INTER_LINEAR,borderMode=cv2.BORDER_CONSTANT,borderValue=(92,96,99))
    return out.reshape(-1,3)[:n].astype(np.float32)
LIGHT=norm(np.array([0.45,-0.55,0.7]))
SKY=np.array([205,190,160],np.float32)  # BGR-ish warm sky
def shade_box(nax,sign):
    n=np.zeros(3); n[nax]=sign; return 0.55+0.45*max(0,float(np.dot(n,LIGHT)))
CAR_COL=np.array([150,120,90],np.float32)  # BGR: steel blue

def trace_static(origin, D):
    D=D.reshape(-1,3); o=origin; n=D.shape[0]
    col=np.empty((n,3),np.float32); col[:]=SKY; tbest=np.full(n,np.inf,np.float32)
    dz=D[:,2]; hitg=dz<-1e-6; tg=np.where(hitg,-o[2]/np.where(hitg,dz,-1),np.inf).astype(np.float32)
    gx=(o[0]+tg*D[:,0]).astype(np.float32); gy=(o[1]+tg*D[:,1]).astype(np.float32)
    idx=np.where(hitg)[0]
    gc=ground_color(gx[idx].reshape(-1,1),gy[idx].reshape(-1,1)).reshape(-1,3)
    fog=np.clip((tg[idx]-12)/25,0,0.6)[:,None]; col[idx]=gc*(1-fog)+SKY*fog; tbest[idx]=tg[idx]
    lo=np.array([-HX,-HY,0.0]); hi=np.array([HX,HY,CAR_H])
    with np.errstate(divide='ignore',invalid='ignore'):
        t1=(lo-o)/D; t2=(hi-o)/D
    tmin=np.minimum(t1,t2); tmax=np.maximum(t1,t2)
    tn=np.nanmax(tmin,axis=1); tf=np.nanmin(tmax,axis=1); ax=np.nanargmax(tmin,axis=1)
    hb=(tn<tf)&(tn>1e-4)&(tn<tbest)
    if hb.any():
        k=hb.sum(); axs=ax[hb]; sgn=np.sign(-D[hb,axs])
        nrm=np.zeros((k,3),np.float32); nrm[np.arange(k),axs]=sgn; f=(0.55+0.45*np.clip(nrm@LIGHT,0,None))[:,None]
        zz=o[2]+tn[hb]*D[hb,2]; win=(zz>0.95)&(zz<1.35)&(axs!=2)
        base=np.where(win[:,None],np.array([70,60,55],np.float32),CAR_COL)
        base[axs==2]=np.array([120,95,72],np.float32)
        col[hb]=base*f; tbest[hb]=tn[hb]
    ground=hitg&~hb
    return dict(col=col,t=tbest,gx=gx,gy=gy,ground=ground,D=D,o=o)
def body_parts(ob):
    if ob.get('kind')=='Trolley':
        x,y=ob['x'],ob['y']; g=(150,150,155)
        return [(x,y,0.3,0.12,1.0,(60,150,215)),(x-0.2,y-0.2,0.05,0,0.12,g),(x+0.2,y-0.2,0.05,0,0.12,g),(x-0.2,y+0.2,0.05,0,0.12,g),(x+0.2,y+0.2,0.05,0,0.12,g)]
    if 'bands' not in ob or ob.get('pole'): return [(ob['x'],ob['y'],ob['r'],0,ob['h'],(60,200,230))]
    k=ob['h']/1.75; x,y=ob['x'],ob['y']; B=ob['bands']
    leg,shirt,skin=B[1][2],B[2][2],B[3][2]
    return [(x-0.1*k,y,0.08*k,0,0.86*k,leg),(x+0.1*k,y,0.08*k,0,0.86*k,leg),
            (x,y,0.19*k,0.84*k,1.46*k,shirt),(x-0.25*k,y,0.055*k,0.9*k,1.44*k,shirt),(x+0.25*k,y,0.055*k,0.9*k,1.44*k,shirt),
            (x,y,0.05*k,1.44*k,1.52*k,skin),(x,y,0.1*k,1.52*k,1.75*k,skin)]
def trace_dynamic(st, objs, shape):
    col=st['col'].copy(); tb=st['t'].copy(); D=st['D']; o=st['o']
    g=st['ground']; gx=st['gx']; gy=st['gy']
    L2=np.array([LIGHT[0],LIGHT[1]]); L2=L2/np.linalg.norm(L2)
    for ob in objs:
        px=gx-ob['x']; py=gy-ob['y']; s=-(px*L2[0]+py*L2[1]); p=np.abs(px*L2[1]-py*L2[0])
        m=g&(s>-ob['r'])&(s<ob['h']*0.7)&(p<ob['r']*1.1); col[m]*=0.62
    for ob in objs:
        for pt in body_parts(ob):
            ox=o[0]-pt[0]; oy=o[1]-pt[1]; r=pt[2]; a=D[:,0]**2+D[:,1]**2; b=2*(ox*D[:,0]+oy*D[:,1]); c=ox*ox+oy*oy-r*r
            disc=b*b-4*a*c; ok=disc>0
            t=np.where(ok,(-b-np.sqrt(np.where(ok,disc,0)))/(2*np.where(a>1e-9,a,1)),np.inf)
            z=o[2]+t*D[:,2]; hit=ok&(t>1e-4)&(z>=pt[3])&(z<=pt[4])&(t<tb)
            if hit.any():
                nx=(ox+t[hit]*D[hit,0])/r; ny=(oy+t[hit]*D[hit,1])/r
                f=(0.45+0.55*np.clip(nx*LIGHT[0]+ny*LIGHT[1]+0.3,0,1))[:,None]
                col[hit]=np.array(pt[5],np.float32)*f; tb[hit]=t[hit]
    return np.clip(col,0,255).astype(np.uint8).reshape(*shape,3)

# fisheye
FE_N=400; FE_F=(FE_N/2)/math.radians(96)
def fisheye_dirs(cam):
    u,v=np.meshgrid(np.arange(FE_N)+0.5,np.arange(FE_N)+0.5)
    x=u-FE_N/2; y=v-FE_N/2; r=np.hypot(x,y); th=r/FE_F; ph=np.arctan2(y,x)
    dc=np.stack([np.sin(th)*np.cos(ph),np.sin(th)*np.sin(ph),np.cos(th)],-1)
    return (dc@cam.R).astype(np.float32), th<math.radians(96)
def fisheye_proj(cam,P):
    d=P-cam.pos; dc=d@cam.R.T; th=np.arctan2(np.hypot(dc[...,0],dc[...,1]),dc[...,2]); ph=np.arctan2(dc[...,1],dc[...,0])
    return (FE_N/2+FE_F*th*np.cos(ph)).astype(np.float32),(FE_N/2+FE_F*th*np.sin(ph)).astype(np.float32),th

# ---------------- BEV ----------------
_c=(np.arange(BN)+0.5)/BN*2*BR-BR
BX,BY=np.meshgrid(_c,_c)
BEV_MAP={}; BEV_OK={}
for _k,_cam in CAMS.items():
    P=np.stack([BX,BY,np.zeros_like(BX)],-1)
    mu,mv,th=fisheye_proj(_cam,P); BEV_MAP[_k]=(mu,mv); BEV_OK[_k]=th<math.radians(95)
SEES={'F':BY<=-HY,'B':BY>=HY,'L':BX<=-HX,'R':BX>=HX}
for _k in CAM_IDS: SEES[_k]=SEES[_k]&BEV_OK[_k]
KGRID=np.full(BX.shape,-1); AGRID=np.zeros(BX.shape,np.float32)
for _i,C in enumerate(CORNERS):
    m=(SEES[C['a']])&(SEES[C['b']])&((BX-C['cx'])*C['sx']>=0)&((BY-C['cy'])*C['sy']>=0)
    KGRID[m]=_i; u=(BX-C['cx'])*C['sx']; v=(BY-C['cy'])*C['sy']; AGRID[m]=(np.arctan2(u,v)*DEG)[m]
CARMASK=(np.abs(BX)<HX+0.02)&(np.abs(BY)<HY+0.02)
def sstep_np(e0,e1,x):
    t=np.clip((x-e0)/(e1-e0),0,1); return t*t*(3-2*t)
def cor_mask(c):
    px=BX-c['x1']; py=BY-c['y1']; s=px*c['dx']+py*c['dy']; p=np.abs(px*c['dy']-py*c['dx'])
    return ((s>=0)&(s<=c['len'])&(p<=c['rad']+c['slope']*s))|(np.hypot(px,py)<=c['rad'])
def weight_grid(m):
    W={c:np.zeros(BX.shape,np.float32) for c in CAM_IDS}
    rest=KGRID<0; taken=np.zeros(BX.shape,bool)
    for c in CAM_IDS:
        mm=rest&SEES[c]&~taken; W[c][mm]=1; taken|=mm
    for i,C in enumerate(CORNERS):
        mm=KGRID==i; wa=1-sstep_np(m['seams'][i]-m['band'],m['seams'][i]+m['band'],AGRID[mm])
        W[C['a']][mm]=wa; W[C['b']][mm]=1-wa
    assigned=np.zeros(BX.shape,bool)
    bg=m.get('blobgrid')
    if bg is not None:
        for ci,c in enumerate(CAM_IDS):
            mm=(bg==ci)&SEES[c]
            for k in CAM_IDS: W[k][mm]=0
            W[c][mm]=1; assigned|=mm
    for c in m['corridors']:
        mm=cor_mask(c)&SEES[c['owner']]&~assigned
        for k in CAM_IDS: W[k][mm]=0
        W[c['owner']][mm]=1; assigned|=mm
    return W, assigned
def warp_all(camimgs):
    return {c:cv2.remap(camimgs[c],*BEV_MAP[c],cv2.INTER_LINEAR,borderMode=cv2.BORDER_CONSTANT) for c in CAM_IDS}
def stitch(warps,m):
    W,owned=weight_grid(m); out=np.zeros(BX.shape+(3,),np.float32)
    for c in CAM_IDS:
        out+=W[c][...,None]*warps[c]
    out[CARMASK]=0
    return np.clip(out,0,255).astype(np.uint8), owned
def w2p(x,y): return (int((x+BR)/(2*BR)*BN), int((y+BR)/(2*BR)*BN))

# ================= v2: proposed method with ladder =================
def risk_weight(o):
    base={'Child':1.6,'Pedestrian':1.0,'Pole':0.5,'Trolley':0.8}.get(o.get('kind'),1.0)
    d=math.hypot(max(0,abs(o['x'])-HX),max(0,abs(o['y'])-HY))
    return base*(1.0+1.0/(0.5+d))
def corridors_for(o,owner,m):
    out=[]
    for c in CAM_IDS:
        if not sees(c,o['x'],o['y']): continue
        s=streak(c,o)
        out.append(dict(owner=owner,x1=s['x1'],y1=s['y1'],dx=s['dx'],dy=s['dy'],len=s['len'],
                        rad=o['r']+m.get('m0',M0)+2*m['sigma']+2*o.get('sig_extra',0),slope=o['r']/s['d']+0.02))
    return out
def explain_mask(est,sigma):
    """region explained by tracked objects (their projections from every camera)"""
    mk=np.zeros(BX.shape,bool); dm=dict(sigma=sigma)
    for o in est:
        for c in corridors_for(o,'F',dm): mk|=cor_mask(c)
    return mk
def make_prop(band=6,sigma=0.0):
    m=make_method('dynamic',band,sigma); m['kind']='prop'; m['level']=1; m['owners']={}; m['blobgrid']=None; m['nblobs']=0; m['worst_id']=None
    return m
def update_prop(m,est,dt):
    # step A: behave like a turning straight seam (cheapest option)
    saved=m['corridors']; m['corridors']=[]
    m['kind']='dynamic'; update_method(m,est,dt); m['kind']='prop'
    rel=[o for o in est if relevant(o)]
    if not rel:
        m['level']=1; m['worst_id']=None; return
    NSsave=globals()['NS']
    sc=[eval_obj(m,o)['score'] for o in rel]
    if min(sc)>=m.get('l2thr',L2_THR) and not any(o.get('coasting') for o in rel):
        m['level']=2; m['worst_id']=None; return
    # step B: ownership labeling over objects that have two candidate cameras
    cand={}; occ=m.get('occl') or set()
    for o in rel:
        cams=[c for c in CAM_IDS if sees(c,o['x'],o['y'])]
        vis=[c for c in cams if (o['id'],c) not in occ]
        cand[o['id']]=vis if vis else cams
    multi=[o for o in rel if len(cand[o['id']])>1]
    order=sorted(rel,key=risk_weight,reverse=True)
    best=None
    globals()['NS']=10
    import itertools
    for combo in itertools.product(*[cand[o['id']] for o in multi]):
        own={o['id']:c for o,c in zip(multi,combo)}
        for o in rel:
            if o['id'] not in own: own[o['id']]=cand[o['id']][0]
        cors=[]
        for o in order: cors+=corridors_for(o,own[o['id']],m)
        m['corridors']=cors
        cost=0; worst=(2,None)
        for o in rel:
            r=eval_obj(m,o); w=risk_weight(o); cost+=w*(1-r['score'])
            if r['score']<worst[0]: worst=(r['score'],o['id'])
            st={c:streak(c,o)['d'] for c in cand[o['id']]}
            cost+=m.get('lam_u',LAM_U)*st[own[o['id']]]/min(st.values())
            if m['owners'].get(o['id']) not in (None,own[o['id']]): cost+=m.get('lam_s',LAM_S)
        if best is None or cost<best[0]: best=(cost,own,cors,worst)
    globals()['NS']=NSsave
    _,own,cors,worst=best
    m['corridors']=cors; m['owners'].update(own)
    n_in=len([o for o in rel if len(cand[o['id']])>1])
    resid=min(eval_obj(m,o)['score'] for o in rel)
    coast=[o for o in rel if o.get('coasting')]
    if coast: m['level']=5; m['worst_id']=coast[0]['id']
    elif resid<L5_THR: m['level']=5; m['worst_id']=worst[1]
    else: m['level']=4 if n_in>=2 else 3; m['worst_id']=None

NEAR=np.hypot(np.maximum(0,np.abs(BX)-HX),np.maximum(0,np.abs(BY)-HY))<4.5
def disagreement_blobs(warps,est,sigma,min_area=120,thr_seed=60,thr_grow=30):
    """camera-disagreement check inside overlaps: 3D content that no tracked object explains.
    seeds: strong disagreement near the car; region grows along weaker disagreement (hysteresis)."""
    grid=np.full(BX.shape,-1,np.int8); expl=explain_mask(est,sigma); n=0
    for i,C in enumerate(CORNERS):
        ovf=(KGRID==i)
        if not ovf.any(): continue
        a=cv2.GaussianBlur(warps[C['a']],(15,15),0).astype(np.float32); b=cv2.GaussianBlur(warps[C['b']],(15,15),0).astype(np.float32)
        dif=np.abs(a-b).mean(-1)
        seed=ovf&NEAR&(dif>thr_seed)&~expl
        seed=cv2.morphologyEx(seed.astype(np.uint8),cv2.MORPH_OPEN,np.ones((3,3),np.uint8)).astype(bool)
        if seed.sum()<min_area: continue
        grow=(ovf&(dif>min(thr_grow,thr_seed-5))&~expl).astype(np.uint8)
        k,lab,stats,_=cv2.connectedComponentsWithStats(grow)
        for li in range(1,k):
            comp=lab==li
            if (comp&seed).sum()<min_area: continue
            reg=cv2.dilate(comp.astype(np.uint8),np.ones((15,15),np.uint8)).astype(bool)&ovf
            ys,xs=np.nonzero(comp&seed); px=BX[ys,xs]; py=BY[ys,xs]
            dcar=np.hypot(np.maximum(0,np.abs(px)-HX),np.maximum(0,np.abs(py)-HY)); kmin=np.argmin(dcar)
            bx,by=px[kmin],py[kmin]
            owner=min((C['a'],C['b']),key=lambda c:math.hypot(bx-CAMS[c].pos[0],by-CAMS[c].pos[1]))
            grid[reg]=CAM_IDS.index(owner); n+=1
    return grid,n

# ======== review round 1 additions ========
RGRID=np.zeros(BX.shape,np.float32)
for _i,C in enumerate(CORNERS):
    _m=KGRID==_i; RGRID[_m]=np.hypot((BX-C['cx'])*C['sx'],(BY-C['cy'])*C['sy'])[_m]
_wg_orig=weight_grid
def weight_grid(m):
    cv=m.get('curve')
    if cv is None: return _wg_orig(m)
    saved=m['seams']; W,assigned=None,None
    # temporarily evaluate per-corner curved seams
    Wd={c:np.zeros(BX.shape,np.float32) for c in CAM_IDS}
    rest=KGRID<0; taken=np.zeros(BX.shape,bool)
    for c in CAM_IDS:
        mm=rest&SEES[c]&~taken; Wd[c][mm]=1; taken|=mm
    for i,C in enumerate(CORNERS):
        mm=KGRID==i
        phi=np.interp(RGRID[mm],CURVE_R,cv[i]) if cv[i] is not None else np.full(mm.sum(),m['seams'][i])
        wa=1-sstep_np(phi-m['band'],phi+m['band'],AGRID[mm]); Wd[C['a']][mm]=wa; Wd[C['b']][mm]=1-wa
    return Wd,np.zeros(BX.shape,bool)
def cor_inside_np(c,X,Y):
    px=X-c['x1']; py=Y-c['y1']; s=px*c['dx']+py*c['dy']; p=np.abs(px*c['dy']-py*c['dx'])
    return ((s>=0)&(s<=c['len'])&(p<=c['rad']+c['slope']*s))|(np.hypot(px,py)<=c['rad'])
def make_curve(band=6,sigma=0.0):
    m=make_method('dynamic',band,sigma); m['kind']='curve'; m['curve']=[None]*4; return m
CURVE_STEPS=3
def update_curve(m,est,dt,lam_s=0.004,lam_c=0.0005):
    """B4: mask-aware curved seam. Seam angle is a function of distance from the corner,
    found by dynamic programming that minimises risk-weighted crossings of the projected
    (dilated) object masks of both cameras (the union of footprints), as in object-avoiding
    seam disclosures that allow arbitrary seam shapes."""
    RR,AA=np.meshgrid(CURVE_R,CURVE_A,indexing='ij')
    for k,C in enumerate(CORNERS):
        rel=[o for o in est if k in relevant(o)]
        prev=m['curve'][k] if m['curve'][k] is not None else np.full(len(CURVE_R),45.0)
        if not rel: target=np.full(len(CURVE_R),45.0)
        else:
            X=C['cx']+C['sx']*RR*np.sin(np.radians(AA)); Y=C['cy']+C['sy']*RR*np.cos(np.radians(AA))
            cost=np.zeros(RR.shape)
            for o in rel:
                w=risk_weight(o); ins=np.zeros(RR.shape,bool)
                for cc in corridors_for(o,'F',m): ins|=cor_inside_np(cc,X,Y)
                cost+=w*ins
            nA=len(CURVE_A); nR=len(CURVE_R); V=np.full((nR,nA),1e9); P=np.zeros((nR,nA),int)
            V[0]=cost[0]+lam_c*np.abs(CURVE_A-45)
            for i in range(1,nR):
                best=np.full(nA,1e9); arg=np.zeros(nA,int)
                for d in range(-CURVE_STEPS,CURVE_STEPS+1):
                    src=np.roll(V[i-1],d); idx=(np.arange(nA)-d)
                    valid=(idx>=0)&(idx<nA); cand=np.where(valid,src+lam_s*abs(d)*2.5,1e9)
                    upd=cand<best; best[upd]=cand[upd]; arg[upd]=idx[upd]
                V[i]=best+cost[i]+lam_c*np.abs(CURVE_A-45); P[i]=arg
            j=int(np.argmin(V[-1])); path=np.zeros(nR)
            for i in range(nR-1,-1,-1): path[i]=CURVE_A[j]; j=P[i,j]
            target=path
        step=np.clip(target-prev,-90*dt,90*dt); m['curve'][k]=prev+step
def update_prop_noassign(m,est,dt):
    """A5: dynamic base seam + dilated ownership regions, independent nearest owners (no joint assignment)."""
    m['corridors']=[]; m['kind']='dynamic'; update_method(m,est,dt); m['kind']='prop_na'
    rel=[o for o in est if relevant(o)]
    if not rel: m['level']=1; return
    if min(eval_obj(m,o)['score'] for o in rel)>=0.95 and not any(o.get('coasting') for o in rel): m['level']=2; return
    cor=[]
    for o in sorted(rel,key=risk_weight,reverse=True):
        cams=[c for c in CAM_IDS if sees(c,o['x'],o['y'])]
        vis=[c for c in cams if (o['id'],c) not in (m.get('occl') or set())]; cams=vis if vis else cams
        st={c:streak(c,o)['d'] for c in cams}; own=min(cams,key=lambda c:st[c])
        prev=m['owners'].get(o['id'])
        if prev in cams and prev!=own and st[prev]<st[own]*1.15: own=prev
        m['owners'][o['id']]=own; cor+=corridors_for(o,own,m)
    m['corridors']=cor; m['level']=3
# ---------- pixel-level (rendered) evaluation ----------
def trace_ids(st,objs):
    tb=st['t'].copy(); D=st['D']; o=st['o']; ids=np.full(tb.shape,-1,np.int16)
    for oi,ob in enumerate(objs):
        for pt in body_parts(ob):
            ox=o[0]-pt[0]; oy=o[1]-pt[1]; r=pt[2]; a=D[:,0]**2+D[:,1]**2; b=2*(ox*D[:,0]+oy*D[:,1]); c=ox*ox+oy*oy-r*r
            disc=b*b-4*a*c; ok=disc>0
            t=np.where(ok,(-b-np.sqrt(np.where(ok,disc,0)))/(2*np.where(a>1e-9,a,1)),np.inf)
            z=o[2]+t*D[:,2]; hit=ok&(t>1e-4)&(z>=pt[3])&(z<=pt[4])&(t<tb)
            tb[hit]=t[hit]; ids[hit]=oi
    return ids
def pixel_masks(stat,objs):
    """per camera: top-view pixels at which that camera's projected image shows object o"""
    M={}
    for c in CAM_IDS:
        idimg=trace_ids(stat[c][0],objs).reshape(FE_N,FE_N).astype(np.float32); idimg[~stat[c][1]]=-1
        w=cv2.remap(idimg,*BEV_MAP[c],cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT,borderValue=-1)
        w[~SEES[c]]=-1; w[CARMASK]=-1; M[c]=w
    return M
def pixel_eval(M,W,objs):
    res=[]
    for oi,o in enumerate(objs):
        vals=[]
        for c in CAM_IDS:
            mk=M[c]==oi; n=mk.sum()
            if n<15: continue
            vals.append(float(W[c][mk].sum()/n))
        vals.sort(reverse=True); b=vals[0] if vals else 0; s2=vals[1] if len(vals)>1 else 0
        res.append(dict(best=b,second=s2,score=max(0,min(1,b-0.5*s2)),hidden=not vals))
    return res

# ---------- occlusion awareness (added after pixel-level evaluation, review round 1) ----------
def occlusions(est):
    """(object id, camera) pairs where another estimated object blocks the camera's line of sight"""
    occ=set()
    for o in est:
        for c in CAM_IDS:
            if not sees(c,o['x'],o['y']): continue
            p=CAMS[c].pos[:2]; v=np.array([o['x'],o['y']])-p; d=np.linalg.norm(v); u=v/d
            for j in est:
                if j is o or j['r']<0.5*o['r'] or j['h']<0.5*o['h']: continue
                w=np.array([j['x'],j['y']])-p; t=float(w@u); lat=abs(w[0]*u[1]-w[1]*u[0])
                if 0<t<d-0.1 and lat<j['r']+0.6*o['r']: occ.add((o['id'],c)); break
    return occ
_eval_orig=eval_obj
def eval_obj(m,o):
    occ=m.get('occl')
    if not occ: return _eval_orig(m,o)
    lst=[]
    for c in CAM_IDS:
        if not sees(c,o['x'],o['y']) or (o['id'],c) in occ: continue
        s=streak(c,o); acc=0; cnt=0
        for i in range(NS):
            t=(i+0.5)/NS; px=s['x1']+(s['x2']-s['x1'])*t; py=s['y1']+(s['y2']-s['y1'])*t
            if abs(px)>BR or abs(py)>BR: continue
            cnt+=1
            if sees(c,px,py): acc+=weights(m,px,py)[c]
        lst.append((acc/max(cnt,1),c))
    lst.sort(reverse=True)
    best=lst[0][0] if lst else 0; second=lst[1][0] if len(lst)>1 else 0
    return dict(best=best,second=second,score=max(0,min(1,best-0.5*second)),owner=lst[0][1] if lst else None)

def cam_visibility(stat,objs,frac=0.4,minpx=30):
    """simulated per-camera detection: object o is detected in camera c if at least `frac` of its
    unoccluded fisheye pixels are visible (occlusion by other objects) and >= minpx pixels"""
    occ=set()
    for c in CAM_IDS:
        st=stat[c][0]; valid=stat[c][1].ravel()
        D=st['D']; o0=st['o']; tb=st['t'].copy(); ids=np.full(tb.shape,-1,np.int16); alone=np.zeros(len(objs))
        for oi,ob in enumerate(objs):
            hit_any=np.zeros(tb.shape,bool)
            for pt in body_parts(ob):
                ox=o0[0]-pt[0]; oy=o0[1]-pt[1]; r=pt[2]; a=D[:,0]**2+D[:,1]**2; b=2*(ox*D[:,0]+oy*D[:,1]); cc=ox*ox+oy*oy-r*r
                disc=b*b-4*a*cc; ok=disc>0
                t=np.where(ok,(-b-np.sqrt(np.where(ok,disc,0)))/(2*np.where(a>1e-9,a,1)),np.inf)
                z=o0[2]+t*D[:,2]; h=ok&(t>1e-4)&(z>=pt[3])&(z<=pt[4])&(t<st['t'])&valid
                hit_any|=h; closer=h&(t<tb); tb[closer]=t[closer]; ids[closer]=oi
            alone[oi]=hit_any.sum()
        for oi,ob in enumerate(objs):
            if not sees(c,ob['x'],ob['y']): continue
            v=(ids==oi).sum()
            if alone[oi]<minpx or v<max(minpx,frac*alone[oi]): occ.add((ob['id'],c))
    return occ

In [ ]:
%%writefile scen.py
import math
PED_B=[(0,0.1,(40,40,45)),(0.1,0.4,(110,60,50)),(0.4,1.5,(30,110,235)),(1.5,1.75,(130,160,200))]
CHILD_B=[(0,0.08,(40,40,45)),(0.08,0.3,(120,120,125)),(0.3,0.9,(60,200,120)),(0.9,1.1,(130,160,200))]
def ped(id,name,x,y): return dict(id=id,name=name,kind='Pedestrian',x=x,y=y,h=1.75,r=0.25,bands=PED_B,nx=0.0,ny=0.0)
def child(id,name,x,y): return dict(id=id,name=name,kind='Child',x=x,y=y,h=1.1,r=0.2,bands=CHILD_B,nx=0.0,ny=0.0)
def lerp(a,b,u): return a+(b-a)*u
def ease(u): u=max(0,min(1,u)); return u*u*(3-2*u)
def sA_make(): return [ped('p1','Pedestrian',-4.2,-2.0)]
def sA_step(o,t):
    u=min(1,t/9.0); o[0]['x']=lerp(-4.2,-0.4,u); o[0]['y']=lerp(-2.0,-3.9,u)
def sB_make(): return [child('c1','Child',-3.2,-4.2)]
def sB_step(o,t):
    u=ease(t/4.0); o[0]['x']=lerp(-3.2,-1.14,u)+0.05*math.sin(t*0.9)*(t>4); o[0]['y']=lerp(-4.2,-2.48,u)
def sC_make(): return [child('c1','Child',-1.45,-2.55),ped('p2','Pedestrian',-1.7,-6.8)]
def sC_step(o,t):
    o[0]['x']=-1.45+0.04*math.sin(t); u=ease(t/5.0); o[1]['x']=lerp(-1.7,-1.45,u); o[1]['y']=lerp(-6.8,-4.0,u)
SCENES=[
 dict(title='Pedestrian walking past the front-left corner',dur=9.5,make=sA_make,step=sA_step,
      text=['A pedestrian crosses the zone that both the front','and the left camera see.',
            'The fixed line crops him as he passes the corner.','The turning line copes; the proposed merge keeps','him whole the entire time.']),
 dict(title='Child standing right at the corner',dur=9.0,make=sB_make,step=sB_step,
      text=['A child walks up and stops close to the bumper.','No straight line can avoid her here: the fixed','line loses her completely and the turning line','hits its angle limit.','The proposed merge bends around her.']),
 dict(title='Two people in the same corner zone',dur=10.0,make=sC_make,step=sC_step,
      text=['A child at the corner and an adult approaching.','Each needs the straight line turned a different','way, so the turning line sacrifices one of them.','The proposed merge shapes around both.']),
]

In [ ]:
%%writefile e2.py
# E2: Monte Carlo benchmark on random corner scenes (analytic footprint model)
import sim2 as sm, numpy as np, math, json, sys, time
sm.NS=12
KINDS=[('Child',1.1,0.2),('Pedestrian',1.75,0.25),('Pole',2.2,0.09)]
def make_scene(rng,k):
    objs=[]
    while len(objs)<k:
        x=rng.uniform(-4.5,-1.05); y=rng.uniform(-5.5,-2.35)
        if any(math.hypot(x-o['x'],y-o['y'])<0.6 for o in objs): continue
        kind,h,r=KINDS[rng.choice(3,p=[0.35,0.5,0.15])]
        sp=0 if kind=='Pole' else rng.uniform(0.2,1.3); th=rng.uniform(0,2*math.pi)
        objs.append(dict(id=f'o{len(objs)}',kind=kind,x=x,y=y,h=h,r=r,vx=sp*math.cos(th),vy=sp*math.sin(th)))
    return objs
def step(objs,dt):
    for o in objs:
        o['x']+=o['vx']*dt; o['y']+=o['vy']*dt
        if not -4.8<o['x']<-1.0: o['vx']*=-1; o['x']=min(max(o['x'],-4.8),-1.0)
        if not -5.8<o['y']<-2.35: o['vy']*=-1; o['y']=min(max(o['y'],-5.8),-2.35)
class Percept:
    def __init__(s,rng,sigma,lat,pdrop,pred=True):
        s.pred=pred; s.rng=rng; s.sig=sigma; s.lat=lat; s.p=pdrop; s.hist={}; s.n={}; s.lost={}; s.prev={}
    def __call__(s,objs,t,dt):
        est=[]
        for o in objs:
            h=s.hist.setdefault(o['id'],[]); h.append((t,o['x'],o['y']))
            nx,ny=s.n.get(o['id'],(0.0,0.0)); tau=0.6
            nx+=-nx/tau*dt+s.sig*math.sqrt(2*dt/tau)*s.rng.normal(); ny+=-ny/tau*dt+s.sig*math.sqrt(2*dt/tau)*s.rng.normal(); s.n[o['id']]=(nx,ny)
            L=s.lost.get(o['id'],0)
            if L<=t and s.rng.random()<s.p*dt: s.lost[o['id']]=t+1.0; L=t+1.0
            coast=L>t and o['id'] in s.prev
            if coast:
                pv=s.prev[o['id']]; e=dict(o,x=pv['x']+pv['evx']*dt,y=pv['y']+pv['evy']*dt,coasting=True,sig_extra=min(0.5,pv.get('sig_extra',0)+0.03)); e['evx'],e['evy']=pv['evx'],pv['evy']
            else:
                td=t-s.lat; past=[p for p in h if p[0]<=td+1e-9] or [h[0]]; _,px,py=past[-1]
                p2=[p for p in h if p[0]<=td-0.2+1e-9] or [h[0]]; _,qx,qy=p2[-1]
                vx,vy=(px-qx)/0.2,(py-qy)/0.2
                k=s.lat if s.pred else 0.0
                e=dict(o,x=px+nx+vx*k,y=py+ny+vy*k,coasting=False,sig_extra=0.0); e['evx'],e['evy']=vx,vy
            s.prev[o['id']]=e; est.append(e)
        return est
METHODS={
 'fixed':lambda sg:sm.make_method('fixed',6,sg),
 'dynamic':lambda sg:sm.make_method('dynamic',6,sg),
 'osm_naive':lambda sg:sm.make_method('shape',6,0.0),
 'uaosm_noassign':lambda sg:sm.make_method('shape',6,sg),
 'uaosm_nodilate':lambda sg:sm.make_prop(6,0.0),
 'uaosm':lambda sg:sm.make_prop(6,sg)}
def run(cond,methods,nscenes,kset,sigma,lat,pdrop,T=4.0,dt=0.1,seed=0,pred=True):
    out={m:dict(ovi=0,lost=0,cut=0,frames=0,switch=0,objsec=0,motion=0,l5=0,sc_ovi=[],sc_lost=[]) for m in methods}
    for sc in range(nscenes):
        k=kset[sc%len(kset)]
        for mname in methods:
            rng=np.random.default_rng(seed*1000+sc); objs=make_scene(rng,k); per=Percept(np.random.default_rng(seed*1000+sc+7),sigma,lat,pdrop,pred)
            m=METHODS[mname](sigma); t=0; prevown={}; so=0; sl=0; sf=0
            while t<T-1e-9:
                step(objs,dt); est=per(objs,t,dt)
                if m['kind']=='prop': sm.update_prop(m,est,dt)
                else: sm.update_method(m,est,dt)
                rs=[sm.eval_obj(m,o) for o in objs]; R=out[mname]
                w=min(r['score'] for r in rs); R['ovi']+=w; R['frames']+=1; so+=w; sf+=1; sl+=any(sm.status(r)=='LOST' for r in rs)
                R['lost']+=any(sm.status(r)=='LOST' for r in rs); R['cut']+=any(sm.status(r) in ('CUT','LOST') for r in rs)
                R['motion']+=m.get('motion',0) if mname=='dynamic' else 0
                R['l5']+=(m.get('level')==5)
                own=dict(m.get('owners',{}))
                R['switch']+=sum(1 for i,c in own.items() if i in prevown and prevown[i]!=c); prevown=own
                R['objsec']+=len(objs)*dt
                t+=dt
            out[mname]['sc_ovi'].append(so/sf); out[mname]['sc_lost'].append(sl/sf)
    for mname,R in out.items():
        f=R['frames']; R.update(ovi=R['ovi']/f,lost=R['lost']/f,cut=R['cut']/f,motion=R['motion']/f,l5=R['l5']/f,switch_per_objs=R['switch']/max(R['objsec'],1e-9))
    return out
if __name__=='__main__':
    cond=sys.argv[1]; t0=time.time()
    C=json.loads(sys.argv[2])
    res=run(cond,**C); res['_cfg']=C; res['_time']=round(time.time()-t0,1)
    json.dump(res,open(f'e2_{cond}.json','w'),indent=1)
    print(cond,res['_time'],{m:(round(v['ovi'],3),round(v['lost'],3)) for m,v in res.items() if not m.startswith('_')})

In [ ]:
%%writefile e3.py
# E3: unknown-object rule (rendered images) and false alarms under extrinsic calibration error
import sim2 as sm, numpy as np, math, json, cv2, sys
STAT={}
for c,cam in sm.CAMS.items():
    D,valid=sm.fisheye_dirs(cam); STAT[c]=(sm.trace_static(cam.pos,D),valid)
def render(objs,stat=STAT):
    cams={}
    for c in sm.CAM_IDS:
        im=sm.trace_dynamic(stat[c][0],objs,(sm.FE_N,sm.FE_N)); im[~stat[c][1]]=0; cams[c]=im
    return sm.warp_all(cams)
mode=sys.argv[1] if len(sys.argv)>1 and __name__=='__main__' else 'none'; rng=np.random.default_rng(11); out={}
if mode=='unknown':
    rows=[]
    for i in range(40):
        while True:
            x=rng.uniform(-3.5,-1.1); y=rng.uniform(-4.5,-2.4)
            if math.hypot(x+0.95,y+2.3)>0.35: break
        kind=['Trolley','Pedestrian','Child'][i%3]
        h,r={'Trolley':(1.0,0.3),'Pedestrian':(1.75,0.25),'Child':(1.1,0.2)}[kind]
        from scen import PED_B, CHILD_B
        o=dict(id='u',kind=kind,x=x,y=y,h=h,r=r,bands=PED_B if kind=='Pedestrian' else CHILD_B)
        w=render([o]); res={}
        for name in ('fixed','dynamic','uaosm_norule','uaosm'):
            m=sm.make_method('fixed',6,0.08) if name=='fixed' else (sm.make_method('dynamic',6,0.08) if name=='dynamic' else sm.make_prop(6,0.08))
            if name=='uaosm':
                g,n=sm.disagreement_blobs(w,[],0.08); m['blobgrid']=g if n else None
            for _ in range(20):   # undetected object: detections list is empty
                (sm.update_prop if m['kind']=='prop' else sm.update_method)(m,[],0.1)
            res[name]=sm.eval_obj(m,o)['best']
        rows.append(dict(kind=kind,x=x,y=y,**res))
    out['rows']=rows
    for name in ('fixed','dynamic','uaosm_norule','uaosm'):
        v=np.array([r[name] for r in rows]); out[name]=dict(mean=float(v.mean()),lost=float((v<0.55).mean()))
    print({k:v for k,v in out.items() if k!='rows'})
    json.dump(out,open('e3_unknown.json','w'),indent=1)
elif mode=='calib':
    lv=[0.0,0.25,0.5,1.0,2.0]
    for err in lv:
        fa=[]; area=[]
        for trial in range(8 if err>0 else 1):
            stat=dict(STAT)
            for c in sm.CAM_IDS:
                cam=sm.CAMS[c]
                if err==0: continue
                yaw=math.radians(rng.normal(0,err)); pit=math.radians(rng.normal(0,err))
                f=cam.R[2].copy()
                Rz=np.array([[math.cos(yaw),-math.sin(yaw),0],[math.sin(yaw),math.cos(yaw),0],[0,0,1]])
                f=Rz@f; ax=np.cross(f,[0,0,1]); ax/=np.linalg.norm(ax)
                K=np.array([[0,-ax[2],ax[1]],[ax[2],0,-ax[0]],[-ax[1],ax[0],0]]); Rp=np.eye(3)+math.sin(pit)*K+(1-math.cos(pit))*K@K
                pc=sm.Cam(tuple(cam.pos),Rp@f)
                D,valid=sm.fisheye_dirs(pc); stat[c]=(sm.trace_static(pc.pos,D),valid)
            w=render([],stat); THR=int(sys.argv[2]) if len(sys.argv)>2 else 60; g,n=sm.disagreement_blobs(w,[],0.08,thr_seed=THR,thr_grow=max(15,THR//2))
            fa.append(n); area.append(float((g>=0).sum())*(2*sm.BR/sm.BN)**2)
        out[str(err)]=dict(false_blob_rate=float(np.mean(np.array(fa)>0)),mean_blobs=float(np.mean(fa)),mean_area_m2=float(np.mean(area)))
        print(err,out[str(err)],flush=True)
    json.dump(out,open(f'e3_calib_{sys.argv[2] if len(sys.argv)>2 else 60}.json','w'),indent=1)

In [ ]:
%%writefile e5.py
import sim2 as sm, numpy as np, math, json, sys, time
sm.NS=12
from scen import PED_B, CHILD_B
import e2
KINDS=[('Child',1.1,0.2),('Pedestrian',1.75,0.25),('Pole',2.2,0.09)]
def make_scene(rng,k):
    objs=[]
    while len(objs)<k:
        x=rng.uniform(-4.5,-1.05); y=rng.uniform(-5.5,-2.35)
        if any(math.hypot(x-o['x'],y-o['y'])<0.6 for o in objs): continue
        kind,h,r=KINDS[rng.choice(3,p=[0.35,0.5,0.15])]
        sp=0 if kind=='Pole' else rng.uniform(0.2,1.3); th=rng.uniform(0,2*math.pi)
        o=dict(id=f'o{len(objs)}',kind=kind,x=x,y=y,h=h,r=r,vx=sp*math.cos(th),vy=sp*math.sin(th))
        if kind=='Pole': o['pole']=True
        else: o['bands']=CHILD_B if kind=='Child' else PED_B
        objs.append(o)
    return objs
def mk(name,sg):
    m,up=_mk(name,sg)
    import sim2 as _s
    def upw(mm,est,dt,_u=up):
        if OCCL: mm['occl']=_s.occlusions(est)
        _u(mm,est,dt)
    return m,upw
OCCL=True
def _mk(name,sg):
    return {'fixed':lambda:(sm.make_method('fixed',6,sg),sm.update_method),
     'dynamic':lambda:(sm.make_method('dynamic',6,sg),sm.update_method),
     'osm_naive':lambda:(sm.make_method('shape',6,0.0),sm.update_method),
     'curve':lambda:(sm.make_curve(6,sg),sm.update_curve),
     'a1_noassign_fixedbase':lambda:(sm.make_method('shape',6,sg),sm.update_method),
     'a5_noassign_dynbase':lambda:(sm.make_prop(6,sg),sm.update_prop_noassign),
     'uaosm_nodilate':lambda:(sm.make_prop(6,0.0),sm.update_prop),
     'uaosm':lambda:(sm.make_prop(6,sg),sm.update_prop),
     'uaosm_novis':lambda:(sm.make_prop(6,sg),sm.update_prop)}[name]()
def run(methods,sc0,nsc,kset,sigma,lat,pdrop,pixel=False,T=4.0,dt=0.1,seed=1,every=2):
    if pixel: from e3 import STAT
    out={m:dict(sc_ovi=[],sc_lost=[],sc_k=[],px_ovi=[],px_lost=[],an_pairs=[],l5=0,frames=0) for m in methods}
    for sc in range(sc0,sc0+nsc):
        k=kset[sc%len(kset)]
        for mname in methods:
            rng=np.random.default_rng(seed*1000+sc); objs=make_scene(rng,k)
            per=e2.Percept(np.random.default_rng(seed*1000+sc+7),sigma,lat,pdrop)
            m,up=mk(mname,sigma); t=0; so=sl=sf=0; po=pl=pf=0; fi=0
            while t<T-1e-9:
                e2.step(objs,dt); est=per(objs,t,dt); up(m,est,dt)
                rs=[sm.eval_obj(m,o) for o in objs]; w=min(r['score'] for r in rs)
                so+=w; sf+=1; sl+=any(sm.status(r)=='LOST' for r in rs)
                out[mname]['l5']+=(m.get('level')==5); out[mname]['frames']+=1
                if pixel and fi%every==0:
                    M=sm.pixel_masks(STAT,objs); W,_=sm.weight_grid(m); pr=sm.pixel_eval(M,W,objs)
                    pw=min(r['score'] for r in pr); po+=pw; pf+=1; pl+=any(sm.status(r)=='LOST' for r in pr)
                    out[mname]['an_pairs'].append((w,pw))
                t+=dt; fi+=1
            R=out[mname]; R['sc_ovi'].append(so/sf); R['sc_lost'].append(sl/sf); R['sc_k'].append(k)
            if pixel: R['px_ovi'].append(po/pf); R['px_lost'].append(pl/pf)
    return out
if __name__=='__main__':
    tag=sys.argv[1]; C=json.loads(sys.argv[2]); t0=time.time()
    res=run(**C); res['_cfg']=C; json.dump(res,open(f'r1_{tag}.json','w'))
    print(tag,round(time.time()-t0),{m:(round(np.mean(v['sc_ovi']),3),round(100*np.mean(v['sc_lost']),1)) for m,v in res.items() if not m.startswith('_')},
          {m:(round(np.mean(v['px_ovi']),3),round(100*np.mean(v['px_lost']),1)) for m,v in res.items() if not m.startswith('_') and v['px_ovi']})

In [ ]:
%%writefile e6.py
import sim2 as sm, numpy as np, json, sys, time
import e5, e2
from e3 import STAT
VIS=True
VISMETH={'uaosm','a5_noassign_dynbase'}
import sim2 as _sm
def _lo(stat,step=3):
    out={}
    for c,(st,valid) in stat.items():
        g=np.zeros((_sm.FE_N,_sm.FE_N),bool); g[::step,::step]=True; ix=np.nonzero(g.ravel())[0]
        out[c]=(dict(D=st['D'][ix],t=st['t'][ix],o=st['o']),valid.ravel()[ix])
    return out
STATLO=_lo(STAT)
def run_px(methods,sc0,nsc,kset,sigma,lat,pdrop,T=4.0,dt=0.1,seed=1,every=2,pred=True):
    out={m:dict(sc_an=[],sc_px=[],sc_an_lost=[],sc_px_lost=[],sc_k=[],pairs=[]) for m in methods}
    for sc in range(sc0,sc0+nsc):
        k=kset[sc%len(kset)]; rng=np.random.default_rng(seed*1000+sc); objs=e5.make_scene(rng,k)
        ms={}; pers={}
        for mn in methods: ms[mn]=e5.mk(mn.replace('+vis',''),sigma); pers[mn]=e2.Percept(np.random.default_rng(seed*1000+sc+7),sigma,lat,pdrop,pred)
        acc={mn:[0,0,0,0,0,0] for mn in methods}; t=0; fi=0; occ=set(); hyst={}
        while t<T-1e-9:
            e2.step(objs,dt)
            if VIS and fi%every==0:
                raw=sm.cam_visibility(STATLO,objs,minpx=4)
                for o in objs:
                    for c in sm.CAM_IDS:
                        key=(o['id'],c); hs=hyst.get(key,[False,0]); obs=key in raw
                        if obs!=hs[0]: hs[1]+=1
                        else: hs[1]=0
                        if hs[1]>=2: hs[0]=obs; hs[1]=0
                        hyst[key]=hs
                occ={k2 for k2,v in hyst.items() if v[0]}
            elif not VIS: occ=set()
            for mn in methods:
                m,up=ms[mn]; m['occl']=occ if mn.endswith('+vis') else set(); up(m,pers[mn](objs,t,dt),dt)
            if fi%every==0:
                M=sm.pixel_masks(STAT,objs)
                for mn in methods:
                    m,_=ms[mn]; rs=[sm.eval_obj(m,o) for o in objs]; W,_=sm.weight_grid(m); pr=sm.pixel_eval(M,W,objs)
                    vis=[i for i,r in enumerate(pr) if not r['hidden']]
                    if not vis: continue
                    a=min(rs[i]['score'] for i in vis); p=min(pr[i]['score'] for i in vis); A=acc[mn]
                    A[0]+=a; A[1]+=p; A[2]+=any(sm.status(rs[i])=='LOST' for i in vis); A[3]+=any(sm.status(pr[i])=='LOST' for i in vis); A[4]+=1
                    A[5]+=len(pr)-len(vis)
                    out[mn]['pairs'].append((a,p))
            t+=dt; fi+=1
        for mn in methods:
            A=acc[mn]; R=out[mn]; R['sc_an'].append(A[0]/A[4]); R['sc_px'].append(A[1]/A[4]); R['sc_an_lost'].append(A[2]/A[4]); R['sc_px_lost'].append(A[3]/A[4]); R['sc_k'].append(k); R.setdefault('hidden',0); R['hidden']+=A[5]
    return out
if __name__=='__main__':
    import e5 as _e5
    _e5.OCCL=False
    if len(sys.argv)>3 and sys.argv[3]=='novis': VIS=False
    tag=sys.argv[1]; C=json.loads(sys.argv[2]); t0=time.time(); res=run_px(**C); json.dump(res,open(f'r1_{tag}.json','w'))
    print(tag,round(time.time()-t0),{m:(round(np.mean(v['sc_an']),3),round(np.mean(v['sc_px']),3),round(100*np.mean(v['sc_px_lost']),1)) for m,v in res.items()})

In [ ]:
%%writefile e9.py
# review round 2 harness: parameter overrides, L5 logging, perception faults, rear corner + ego motion, covariance miscalibration
import sim2 as sm, numpy as np, json, sys, time, math
import e5, e2, e6
e5.OCCL=False
from e3 import STAT
from scen import PED_B, CHILD_B
def make_scene(rng,k,region):
    objs=e5.make_scene(rng,k)
    if region=='RL':
        for o in objs: o['y']=-o['y']; o['vy']=-o['vy']
    return objs
def step(objs,dt,region,vego):
    ylo,yhi=(-5.8,-2.35) if region=='FL' else (2.35,5.8)
    for o in objs:
        o['x']+=o['vx']*dt; o['y']+=(o['vy']-vego)*dt
        if not -4.8<o['x']<-1.0: o['vx']*=-1; o['x']=min(max(o['x'],-4.8),-1.0)
        if vego==0 and not ylo<o['y']<yhi: o['vy']*=-1; o['y']=min(max(o['y'],ylo),yhi)
def faults(est,rng,t,dt,state,pmis,rfp,region):
    out=[]
    for e in est:
        if pmis>0:
            if e['id'] not in state['mis']: state['mis'][e['id']]=rng.random()<pmis
            if state['mis'][e['id']] and e['kind'] in ('Child','Pedestrian'):
                e=dict(e); e['kind']='Pedestrian' if e['kind']=='Child' else 'Child'; e['h']=1.75 if e['kind']=='Pedestrian' else 1.1
        out.append(e)
    if rfp>0 and rng.random()<rfp*dt:
        y=rng.uniform(-5.5,-2.35); y=y if region=='FL' else -y
        state['fp'].append(dict(id=f'fp{len(state["fp"])}',kind='Pedestrian',x=rng.uniform(-4.5,-1.05),y=y,h=1.75,r=0.25,t_end=t+0.5,coasting=False,sig_extra=0.0))
    out+=[f for f in state['fp'] if f['t_end']>t]
    return out
def run(methods,sc0,nsc,kset,sigma,lat,pdrop,sig_true=None,region='FL',vego=0.0,pmis=0.0,rfp=0.0,params=None,T=4.0,dt=0.1,seed=1,every=2):
    if params:
        for k,v in params.items(): setattr(sm,k,v)
    st_true=sigma if sig_true is None else sig_true
    out={m:dict(px=[],pl=[],l5=0,l5c=0,l5r=0,l5r_ok=0,fr=0,lostfr=0,lost_warned=0,lost_k=[0]*6,k=[]) for m in methods}
    for sc in range(sc0,sc0+nsc):
        k=kset[sc%len(kset)]; rng=np.random.default_rng(seed*1000+sc); objs=make_scene(rng,k,region)
        ms={mn:e5.mk(mn.replace('+vis',''),sigma) for mn in methods}
        pers={mn:e2.Percept(np.random.default_rng(seed*1000+sc+7),st_true,lat,pdrop) for mn in methods}
        fst={mn:dict(mis={},fp=[]) for mn in methods}; frng={mn:np.random.default_rng(seed*1000+sc+99) for mn in methods}
        acc={mn:[0,0,0] for mn in methods}; t=0; fi=0; occ=set(); hyst={}
        while t<T-1e-9:
            step(objs,dt,region,vego)
            if fi%every==0:
                raw=sm.cam_visibility(e6.STATLO,objs,minpx=4)
                for o in objs:
                    for c in sm.CAM_IDS:
                        key=(o['id'],c); hs=hyst.get(key,[False,0]); obs=key in raw
                        hs[1]=hs[1]+1 if obs!=hs[0] else 0
                        if hs[1]>=2: hs[0]=obs; hs[1]=0
                        hyst[key]=hs
                occ={k2 for k2,v in hyst.items() if v[0]}
            for mn in methods:
                m,up=ms[mn]; m['occl']=occ if mn.endswith('+vis') else set()
                est=faults(pers[mn](objs,t,dt),frng[mn],t,dt,fst[mn],pmis,rfp,region); up(m,est,dt)
            if fi%every==0:
                M=sm.pixel_masks(STAT,objs)
                for mn in methods:
                    m,_=ms[mn]; W,_=sm.weight_grid(m); pr=sm.pixel_eval(M,W,objs); vis=[r for r in pr if not r['hidden']]
                    R=out[mn]; R['fr']+=1
                    lost=any(sm.status(r)=='LOST' for r in vis) if vis else False
                    if vis: A=acc[mn]; A[0]+=min(r['score'] for r in vis); A[1]+=lost; A[2]+=1
                    if lost: R['lostfr']+=1; R['lost_warned']+=(m.get('level')==5); R['lost_k'][k]+=1
                    if m.get('level')==5:
                        R['l5']+=1; coast=any(e.get('coasting') for e in pers[mn].prev.values())
                        if coast: R['l5c']+=1
                        else:
                            R['l5r']+=1; R['l5r_ok']+=(not lost)
            t+=dt; fi+=1
        for mn in methods:
            A=acc[mn]
            if A[2]: out[mn]['px'].append(A[0]/A[2]); out[mn]['pl'].append(A[1]/A[2]); out[mn]['k'].append(k)
    return out
if __name__=='__main__':
    tag=sys.argv[1]; C=json.loads(sys.argv[2]); t0=time.time(); r=run(**C); json.dump(r,open(f'r2_{tag}.json','w'))
    print(tag,round(time.time()-t0),{m:dict(px=round(np.mean(v['px']),3),lost=round(100*np.mean(v['pl']),1),L5=round(100*v['l5']/max(v['fr'],1),1),L5coast=round(100*v['l5c']/max(v['fr'],1),1),L5res=round(100*v['l5r']/max(v['fr'],1),1),L5res_nolost=v['l5r_ok'],lost_frames=v['lostfr'],lost_warned=v['lost_warned']) for m,v in r.items()})

In [ ]:
%%writefile cond.py
# Scenario catalog: time of day x weather -> assumed perception degradation (documented in the paper)
import e9, json, sys, time, numpy as np
TOD={'morning':dict(sigma=0.08,pdrop=0.03,pmis=0.05,rfp=0.10),   # low sun, long shadows, glare
     'afternoon':dict(sigma=0.06,pdrop=0.01,pmis=0.03,rfp=0.05), # high sun, best case
     'evening':dict(sigma=0.10,pdrop=0.04,pmis=0.08,rfp=0.15),   # blue hour, mixed artificial light
     'night':dict(sigma=0.14,pdrop=0.08,pmis=0.12,rfp=0.30)}     # artificial light only, noise
WX={'clear':dict(sm=1.0,dp=0.0,dm=0.0,df=0.0),
    'rain':dict(sm=1.4,dp=0.04,dm=0.05,df=0.30),                 # droplets, spray, wet reflections
    'fog':dict(sm=1.6,dp=0.06,dm=0.08,df=0.10)}                  # reduced range and contrast
def params(t,w):
    a=TOD[t]; b=WX[w]
    return dict(sigma=round(a['sigma']*b['sm'],3),pdrop=round(a['pdrop']+b['dp'],3),pmis=round(a['pmis']+b['dm'],3),rfp=round(a['rfp']+b['df'],3))
if __name__=='__main__':
    for key in sys.argv[1:]:
        t,w=key.split('-'); P=params(t,w); t0=time.time()
        r=e9.run(methods=['dynamic','uaosm+vis'],sc0=500,nsc=10,kset=[1,2,3,4,5],lat=0.1,**P)
        json.dump(dict(params=P,res=r),open(f'cond_{key}.json','w'))
        print(key,P,round(time.time()-t0),{m:round(100*np.mean(v['pl']),1) for m,v in r.items()},'L5',round(100*r['uaosm+vis']['l5']/max(r['uaosm+vis']['fr'],1),1),flush=True)

In [ ]:
import importlib
import sim2 as sm, e2, e5
e5.OCCL = False                      # owners use the per-camera visibility set, not a geometric predictor
from e3 import STAT, render           # static fisheye renderings (takes a few seconds)
import e6, e9, cond
e9.e5.OCCL = False
print('Library loaded. Cameras:', list(sm.CAMS), '| top view', sm.BN, 'px over ±', sm.BR, 'm')

## 3. Geometry, footprint model and OVI (Section III)
A point at height $z$ seen by camera $c$ at height $h_c$ lands on the ground at
$\pi_c(\mathbf X)=\mathbf p_c+\frac{h_c}{h_c-z}(\mathbf X_{xy}-\mathbf p_c)$ (Eq. 1).
An upright object therefore projects to a wedge along the camera ray whose length is
$\ell_c=d_c h_o/(h_c-h_o)$ for $h_o<h_c$ and reaches the display edge otherwise (Eq. 2).
**Analytic OVI** (Eq. 4–5) samples this footprint; **pixel-level OVI** (Eq. 6) uses rendered per-object identity images and is the evaluation metric.

In [ ]:
from scen import PED_B, CHILD_B
ped = dict(id='p', kind='Pedestrian', x=-2.2, y=-3.4, h=1.75, r=0.25, bands=PED_B)
for c in ['F','L']:
    s = sm.streak(c, ped); print(f"camera {c}: distance {s['d']:.2f} m, footprint length {s['len']:.2f} m")
imgs = {c: sm.trace_dynamic(STAT[c][0], [ped], (sm.FE_N, sm.FE_N)) for c in sm.CAM_IDS}
fig, ax = plt.subplots(1, 4, figsize=(12, 3))
for a, c in zip(ax, sm.CAM_IDS):
    im = imgs[c].copy(); im[~STAT[c][1]] = 0; a.imshow(im[..., ::-1]); a.set_title(f'fisheye {c}'); a.axis('off')
plt.show()
warps = render([ped])
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, kind in zip(ax, ['fixed', 'dynamic', 'prop']):
    m = sm.make_prop(6, 0.0) if kind == 'prop' else sm.make_method(kind, 6, 0.0)
    for _ in range(25): (sm.update_prop if kind == 'prop' else sm.update_method)(m, [ped], 0.1)
    bev, _ = sm.stitch(warps, m); a.imshow(bev[..., ::-1], extent=[-sm.BR, sm.BR, sm.BR, -sm.BR])
    W, _ = sm.weight_grid(m); px = sm.pixel_eval(sm.pixel_masks(STAT, [ped]), W, [ped])[0]; an = sm.eval_obj(m, ped)
    a.set_title(f"{kind}: analytic OVI {an['score']:.2f}, pixel OVI {px['score']:.2f}"); a.set_xlim(-6, 1); a.set_ylim(0, -7)
plt.show()

## 4. Feasibility of straight seams (Section IV)
**Proposition 1.** A straight seam at angle $\varphi$ (blend half-width $\beta$) preserves object $o$ iff $\varphi\ge A_a(o)+\beta$ or $\varphi\le A_b(o)-\beta$.
**Corollary 1.** With admissible angles $[\varphi_{min},\varphi_{max}]$, $o$ is unpreservable when $A_a+\beta>\varphi_{max}$ and $A_b-\beta<\varphi_{min}$.
**Proposition 2.** For several objects the feasible set is the intersection of the per-object sets, so it can only shrink as objects are added.

In [ ]:
C0 = sm.CORNERS[0]
def alpha(x, y): return math.degrees(math.atan2((x-C0['cx'])*C0['sx'], (y-C0['cy'])*C0['sy']))
def fp_angles(o, c):
    s = sm.streak(c, o); out = []
    for i in range(60):
        t = (i+0.5)/60; x = s['x1']+(s['x2']-s['x1'])*t; y = s['y1']+(s['y2']-s['y1'])*t
        if abs(x) <= sm.BR and abs(y) <= sm.BR and sm.corner_of(x, y) == 0: out.append(alpha(x, y))
    return out
def infeasible_map(h, r, lo, hi, b, step):
    xs = np.arange(-5.0, -0.95, step); ys = np.arange(-5.5, -2.3, step); M = np.zeros((len(ys), len(xs)), bool)
    for j, y in enumerate(ys):
        for i, x in enumerate(xs):
            o = dict(x=x, y=y, h=h, r=r); aF = fp_angles(o, 'F'); aL = fp_angles(o, 'L')
            AF = max(aF) if aF else -1e9; AL = min(aL) if aL else 1e9
            M[j, i] = not (AF+b <= hi or AL-b >= lo)
    X, Y = np.meshgrid(xs, ys); d = np.hypot(X-C0['cx'], Y-C0['cy'])
    return M, xs, ys, {R: 100*M[d <= R].mean() for R in (0.5, 1.0, 1.5, 2.0)}
rows = []
for name, h, r, lo, hi, b in [('child/adult', 1.1, .2, 15, 75, 6), ('0.5 m bollard', .5, .1, 15, 75, 6), ('child/adult', 1.1, .2, 10, 80, 6),
                              ('child/adult', 1.1, .2, 5, 85, 6), ('child/adult', 1.1, .2, 0, 90, 6), ('child/adult', 1.1, .2, 15, 75, 3), ('child/adult', 1.1, .2, 15, 75, 10)]:
    M, xs, ys, fr = infeasible_map(h, r, lo, hi, b, N['grid']); rows.append((name, lo, hi, b, fr))
    if (name, lo, hi, b) == ('child/adult', 15, 75, 6): M1575 = (M, xs, ys)
print(f"{'object':14s} {'seams':8s} beta   <=0.5m  <=1.0m  <=1.5m  <=2.0m")
for name, lo, hi, b, fr in rows: print(f"{name:14s} {lo:>2d}-{hi:<3d}   {b:>2d}°   " + '  '.join(f'{fr[R]:5.1f}%' for R in (0.5, 1.0, 1.5, 2.0)))
print('Paper (Table II):', REF['feasibility'])
M, xs, ys = M1575
plt.figure(figsize=(4, 3.4)); plt.imshow(M[::-1, ::-1], origin='lower', extent=[-(xs[-1]+.95), -(xs[0]+.95), -(ys[-1]+2.3), -(ys[0]+2.3)], cmap='Reds')
plt.xlim(0, 1.4); plt.ylim(0, 1.2); plt.xlabel('lateral from vehicle side (m)'); plt.ylabel('ahead of bumper (m)'); plt.title('Infeasible zone, 15–75°, β=6°'); plt.show()

## 5. Methods compared (Sections V–VI)
| Paper label | Code name | Description |
|---|---|---|
| B1 Fixed seam | `fixed` | 45° diagonal |
| B2 Straight dynamic seam | `dynamic` | angle search 15–75°, rate-limited |
| B3 One camera per object | `osm_naive` | nearest camera owns each object, no dilation |
| B4 Curved mask-avoiding seam | `curve` | dynamic-programming seam avoiding projected masks |
| A5 w/o joint assignment | `a5_noassign_dynbase+vis` | independent nearest visible owner |
| A6 w/o visibility | `uaosm` | owners may be occluded cameras |
| UA-OSM (full) | `uaosm+vis` | visibility set + covariance dilation + joint assignment + ladder |

The suffix `+vis` gives a method the debounced per-camera visibility set (simulated from rendered fisheye images).

## 6. Qualitative result (Fig. 1): crowded corner

In [ ]:
from scen import PED_B, CHILD_B
crowd = [dict(id='c1',kind='Child',x=-1.2,y=-2.55,h=1.1,r=0.2,bands=CHILD_B), dict(id='p2',kind='Pedestrian',x=-1.5,y=-3.9,h=1.75,r=0.25,bands=PED_B),
         dict(id='p3',kind='Pedestrian',x=-2.4,y=-3.0,h=1.75,r=0.25,bands=PED_B), dict(id='q4',kind='Pole',x=-2.7,y=-4.5,h=2.2,r=0.09,pole=True),
         dict(id='c5',kind='Child',x=-2.2,y=-4.1,h=1.1,r=0.2,bands=CHILD_B)]
W8 = render(crowd); Mk = sm.pixel_masks(STAT, crowd); occ = sm.cam_visibility(STAT, crowd)
fig, ax = plt.subplots(1, 3, figsize=(13, 4.3))
for a, (title, m, up) in zip(ax, [('B1 fixed', sm.make_method('fixed',6,0.0), sm.update_method), ('B2 dynamic straight', sm.make_method('dynamic',6,0.0), sm.update_method), ('UA-OSM', sm.make_prop(6,0.0), sm.update_prop)]):
    if title == 'UA-OSM': m['occl'] = occ
    for _ in range(30): up(m, crowd, 0.1)
    bev, _ = sm.stitch(W8, m); a.imshow(bev[..., ::-1], extent=[-sm.BR, sm.BR, sm.BR, -sm.BR])
    W, _ = sm.weight_grid(m); pr = sm.pixel_eval(Mk, W, crowd); st = [sm.status(r) for r in pr]
    for o, s in zip(crowd, st): a.add_patch(plt.Circle((o['x'], o['y']), .32, fill=False, lw=2, ec={'LOST':'r','CUT':'orange','DOUBLED':'orange','VISIBLE':'lime'}[s]))
    a.set_xlim(-6, .6); a.set_ylim(-.8, -7); a.set_title(f"{title}: {st.count('LOST')} lost, {st.count('CUT')} cut"); a.set_xticks([]); a.set_yticks([])
plt.show()

## 7. Main benchmark on rendered pixels (Section VII-A/B)
Random front-left corner scenes with 1–5 objects (children, adults, poles), σ = 8 cm, latency 100 ms, track losses 0.1 s⁻¹.
Statistics: bootstrap 95 % CIs over scenes, paired Wilcoxon signed-rank vs. UA-OSM with Holm correction, matched-pairs rank-biserial effect size.

In [ ]:
from scipy.stats import wilcoxon
METHODS = ['fixed', 'dynamic', 'osm_naive', 'curve', 'a5_noassign_dynbase+vis', 'uaosm', 'uaosm+vis']
LABEL = dict(zip(METHODS, ['B1 Fixed seam', 'B2 Straight dynamic seam', 'B3 One camera per object', 'B4 Curved mask-avoiding seam', 'A5 w/o joint assignment', 'A6 w/o visibility', 'UA-OSM (full)']))
t0 = time.time()
res = {m: dict(px=[], pl=[], an=[], k=[], pairs=[]) for m in METHODS}
for s0 in range(0, N['main'], 5):
    r = e6.run_px(methods=METHODS, sc0=s0, nsc=min(5, N['main']-s0), kset=[1,2,3,4,5], sigma=0.08, lat=0.1, pdrop=0.1)
    for m in METHODS:
        res[m]['px'] += r[m]['sc_px']; res[m]['pl'] += r[m]['sc_px_lost']; res[m]['an'] += r[m]['sc_an']; res[m]['k'] += r[m]['sc_k']; res[m]['pairs'] += r[m]['pairs']
    print(f'{s0+5} scenes done, {time.time()-t0:.0f} s')
rng = np.random.default_rng(0)
def boot(x): x = np.asarray(x); b = [rng.choice(x, len(x)).mean() for _ in range(2000)]; return np.percentile(b, [2.5, 97.5])
print(f"\n{'Method':30s} {'pixel OVI [95% CI]':24s} {'lost frames [95% CI]':24s} r(an,px) | paper (OVI, lost %)")
for m in METHODS:
    R = res[m]; lo, hi = boot(R['px']); llo, lhi = boot(100*np.array(R['pl'])); p = np.array(R['pairs'])
    rr = np.corrcoef(p[:, 0], p[:, 1])[0, 1] if len(p) > 2 and p[:, 1].std() > 0 else float('nan')
    print(f"{LABEL[m]:30s} {np.mean(R['px']):.3f} [{lo:.3f},{hi:.3f}]   {100*np.mean(R['pl']):5.1f}% [{llo:4.1f},{lhi:4.1f}]   {rr:5.2f}   | {REF['main_pixel'][LABEL[m]]}")
ref = np.array(res['uaosm+vis']['px']); tests = []
for m in METHODS[:-1]:
    x = np.array(res[m]['px']); d = ref-x
    try: p = wilcoxon(ref, x).pvalue
    except ValueError: p = 1.0
    nz = d[d != 0]; rk = np.argsort(np.argsort(np.abs(nz)))+1
    rb = (rk[nz > 0].sum()-rk[nz < 0].sum())/max(rk.sum(), 1); tests.append([LABEL[m], p, rb, np.median(d)])
tests.sort(key=lambda t: t[1])
for i, t in enumerate(tests): t.append(min(1, t[1]*(len(tests)-i)))
print('\nUA-OSM vs. each method: p, Holm-adjusted p, rank-biserial r, median paired difference')
for t in tests: print(f'  {t[0]:30s} p={t[1]:.2e}  Holm={t[4]:.2e}  r={t[2]:+.2f}  Δ={t[3]:+.3f}')
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
for m in METHODS:
    if m == 'uaosm': continue
    k = np.array(res[m]['k'])
    ax[0].plot(range(1, 6), [np.mean(np.array(res[m]['px'])[k == kk]) if (k == kk).any() else np.nan for kk in range(1, 6)], 'o-', label=LABEL[m])
    ax[1].plot(range(1, 6), [100*np.mean(np.array(res[m]['pl'])[k == kk]) if (k == kk).any() else np.nan for kk in range(1, 6)], 'o-')
ax[0].set_ylabel('pixel worst-object OVI'); ax[1].set_ylabel('frames with a lost object (%)')
for a in ax: a.set_xlabel('objects in corner zone')
ax[0].legend(fontsize=7); plt.show()
print('Paper per-k lost % (B2, UA-OSM):', REF['perk_lost'])

## 8. Planning-model mismatch, detection error and latency (Sections VII-C/D/E)
The correlation column above quantifies why analytic OVI cannot be used as evidence: UA-OSM plans with it, so its analytic and pixel scores agree only weakly (paper: r = 0.45 vs. 0.93 for the non-planning fixed seam).

In [ ]:
def lost_rate(methods, **kw):
    r = e6.run_px(methods=methods, sc0=200, nsc=N['sweep'], kset=[1,2,3,4,5], **kw)
    return {m: 100*np.mean(r[m]['sc_px_lost']) for m in methods}
print('Detection error sweep (no track losses): B2 | UA-OSM w/o dilation | UA-OSM   — paper:', REF['sigma'])
for s in [0.0, 0.2, 0.3]:
    r = lost_rate(['dynamic', 'uaosm_nodilate+vis', 'uaosm+vis'], sigma=s, lat=0.1, pdrop=0.0)
    print(f'  sigma {s*100:4.0f} cm: ' + '  '.join(f'{v:5.1f}%' for v in r.values()))
print('Latency (σ = 8 cm): B2 | UA-OSM   — paper:', REF['latency'])
for lat, pred in [(0.1, True), (0.3, True), (0.3, False)]:
    r = lost_rate(['dynamic', 'uaosm+vis'], sigma=0.08, lat=lat, pdrop=0.0, pred=pred)
    print(f'  {lat*1000:.0f} ms, prediction {pred}: ' + '  '.join(f'{v:5.1f}%' for v in r.values()))

## 9. Robustness, parameter sensitivity and driver warnings (Section VII)
Warning columns: share of frames at ladder level 5 (L5), split into coasting-track and residual-conflict triggers, and how many lost frames had an active warning.

In [ ]:
def rb(tag, methods=['uaosm+vis'], **kw):
    r = e9.run(methods=methods, sc0=300, nsc=N['robust'], kset=[1,2,3,4,5], lat=0.1, **kw)
    for m, v in r.items():
        fr = max(v['fr'], 1)
        print(f"{tag:42s} {m:10s} lost {100*np.mean(v['pl']):5.1f}%  L5 {100*v['l5']/fr:5.1f}% (coast {100*v['l5c']/fr:4.1f}, other {100*v['l5r']/fr:4.1f})  lost frames warned {v['lost_warned']}/{v['lostfr']}")
rb('default (track loss 0.1/s)', sigma=0.08, pdrop=0.1)
rb('m0 = 0.05 m', sigma=0.08, pdrop=0.1, params={'M0': 0.05}); rb('m0 = 0.20 m', sigma=0.08, pdrop=0.1, params={'M0': 0.2})
rb('lambda_s = 0', sigma=0.08, pdrop=0.1, params={'LAM_S': 0.0}); rb('lambda_s = 1.0', sigma=0.08, pdrop=0.1, params={'LAM_S': 1.0})
e9.sm.M0, e9.sm.LAM_S = 0.12, 0.25
rb('track loss 0.02/s', sigma=0.08, pdrop=0.02)
rb('perception faults: 20% class swaps, 0.5 FP/s', methods=['dynamic', 'uaosm+vis'], sigma=0.08, pdrop=0.02, pmis=0.2, rfp=0.5)
rb('rear-left corner, reversing 0.5 m/s', methods=['dynamic', 'uaosm+vis'], sigma=0.08, pdrop=0.02, region='RL', vego=0.5)
rb('true error 16 cm, assumed 8 cm', sigma=0.08, sig_true=0.16, pdrop=0.02)
rb('true error 16 cm, assumed 16 cm', sigma=0.16, sig_true=0.16, pdrop=0.02)

## 10. Scenario catalog: time of day × weather × density
Conditions act through perception quality (the ray-traced geometry and the identity-based metric are illumination-independent). The degradation values below are engineering **assumptions**, listed in the paper's catalog table.

In [ ]:
print('Assumed perception parameters per condition:')
for t in cond.TOD:
    for w in cond.WX: print(f'  {t:9s} {w:5s}', cond.params(t, w))
T, WX = ['morning', 'afternoon', 'evening', 'night'], ['clear', 'rain', 'fog']
B = np.zeros((3, 4)); U = np.zeros((3, 4)); L5 = np.zeros((3, 4)); kB = {k: [] for k in range(1, 6)}; kU = {k: [] for k in range(1, 6)}
for i, w in enumerate(WX):
    for j, t in enumerate(T):
        r = e9.run(methods=['dynamic', 'uaosm+vis'], sc0=500, nsc=max(N['catalog'], 1), kset=[1,2,3,4,5], lat=0.1, **cond.params(t, w))
        B[i, j] = 100*np.mean(r['dynamic']['pl']); U[i, j] = 100*np.mean(r['uaosm+vis']['pl']); L5[i, j] = 100*r['uaosm+vis']['l5']/max(r['uaosm+vis']['fr'], 1)
        for m, acc in (('dynamic', kB), ('uaosm+vis', kU)):
            for k, v in zip(r[m]['k'], r[m]['pl']): acc[k].append(100*v)
        print(f'{t:9s} {w:5s}  B2 {B[i,j]:5.1f}%  UA-OSM {U[i,j]:5.1f}%  L5 {L5[i,j]:5.1f}%   | paper B2 {REF["catalog_B2"][i][j]:.1f}%  UA {REF["catalog_UA"][i][j]:.1f}%')
fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
for a, M, tt in ((ax[0], B, 'B2 lost frames (%)'), (ax[1], U, 'UA-OSM lost frames (%)')):
    a.imshow(M, cmap='Reds', vmin=0, vmax=30); a.set_xticks(range(4)); a.set_xticklabels(T); a.set_yticks(range(3)); a.set_yticklabels(WX); a.set_title(tt)
    for i in range(3):
        for j in range(4): a.text(j, i, f'{M[i,j]:.1f}', ha='center', va='center')
ax[2].plot(range(1, 6), [np.mean(kB[k]) if kB[k] else np.nan for k in range(1, 6)], 'o-', label='B2')
ax[2].plot(range(1, 6), [np.mean(kU[k]) if kU[k] else np.nan for k in range(1, 6)], 'o-', label='UA-OSM')
ax[2].set_xlabel('objects in corner zone'); ax[2].set_ylabel('lost frames (%)'); ax[2].legend(); plt.show()

## 11. Unexplained-content rule and calibration false alarms (proof of concept)
Undetected objects are withheld from the tracker. The rule protects cross-camera disagreement (Eq. 11). The second cell shows why it is not deployable: small extrinsic calibration errors create false components on empty ground.

In [ ]:
rng2 = np.random.default_rng(11); cases = []
for i in range(N['unknown']):
    while True:
        x = rng2.uniform(-3.5, -1.1); y = rng2.uniform(-4.5, -2.4)
        if math.hypot(x+.95, y+2.3) > .35: break
    kind = ['Trolley', 'Pedestrian', 'Child'][i % 3]; h, r = {'Trolley': (1.0, .3), 'Pedestrian': (1.75, .25), 'Child': (1.1, .2)}[kind]
    cases.append(dict(id='u', kind=kind, x=x, y=y, h=h, r=r, bands=PED_B if kind == 'Pedestrian' else CHILD_B))
Ws = [render([o]) for o in cases]; Ms = [sm.pixel_masks(STAT, [o]) for o in cases]
for thr in (None, 60, 45, 35):
    vals = []
    for o, w, M in zip(cases, Ws, Ms):
        m = sm.make_prop(6, 0.08)
        if thr is not None:
            g, n = sm.disagreement_blobs(w, [], 0.08, thr_seed=thr, thr_grow=max(15, thr//2)); m['blobgrid'] = g if n else None
        for _ in range(20): sm.update_prop(m, [], 0.1)
        Wg, _ = sm.weight_grid(m); vals.append(sm.pixel_eval(M, Wg, [o])[0]['best'])
    v = np.array(vals); print(f"threshold {str(thr):5s}: mean OVI {v.mean():.3f}, lost {100*(v<0.55).mean():5.1f}%   | paper {REF['unknown']['disabled' if thr is None else str(thr)]}")

In [ ]:
def false_components(err_deg, thr, trials):
    rng3 = np.random.default_rng(5); hits = []
    for _ in range(trials):
        stat = dict(STAT)
        for c in sm.CAM_IDS:
            if err_deg == 0: continue
            cam = sm.CAMS[c]; yaw = math.radians(rng3.normal(0, err_deg)); pit = math.radians(rng3.normal(0, err_deg))
            f = cam.R[2].copy(); Rz = np.array([[math.cos(yaw), -math.sin(yaw), 0], [math.sin(yaw), math.cos(yaw), 0], [0, 0, 1]]); f = Rz@f
            axv = np.cross(f, [0, 0, 1]); axv /= np.linalg.norm(axv); K = np.array([[0, -axv[2], axv[1]], [axv[2], 0, -axv[0]], [-axv[1], axv[0], 0]])
            pc = sm.Cam(tuple(cam.pos), (np.eye(3)+math.sin(pit)*K+(1-math.cos(pit))*K@K)@f)
            D, valid = sm.fisheye_dirs(pc); stat[c] = (sm.trace_static(pc.pos, D), valid)
        w = render([], stat); g, n = sm.disagreement_blobs(w, [], 0.08, thr_seed=thr, thr_grow=max(15, thr//2)); hits.append(n > 0)
    return 100*np.mean(hits)
for e in [0.0, 0.5, 1.0]:
    print(f"calibration error {e:.1f}°: trials with a false component — τ=60: {false_components(e, 60, 1 if e == 0 else N['calib']):.0f}%   τ=35: {false_components(e, 35, 1 if e == 0 else N['calib']):.0f}%")

## 12. Real-world appearance check (photographs)
Seven smartphone photographs of the author at dusk/night (26 Sep 2026, 19:50–19:53, dry, floodlights and street lights).
The metric is the rule's own quantity: mean absolute 8-bit RGB difference between blurred clothing and asphalt regions.
**These photos test the appearance assumption only; they do not validate merging.**
Upload your own photos (JPEG) to recompute with your own region boxes, or use the stored measurements.

In [ ]:
stored = REF['photo_contrast']
print(f"{'photo':10s} {'lighting':22s} shirt  jeans  shoes")
for k, v in stored.items():
    print(f"{k:10s} {v['light']:22s} " + '  '.join(f"{v[p]['dk']:5.1f}" if p in v else '   — ' for p in ('shirt', 'jeans', 'shoes')))
print('Thresholds of the rule: 35 / 45 / 60 — lower-body clothing mostly falls below 45.')
try:
    from google.colab import files
    print('Optional: upload photos and edit BOXES below (x0,y0,x1,y1 in pixels).')
    # up = files.upload()
except ImportError:
    pass
from PIL import Image, ImageOps, ImageFilter
def contrast(path, boxes):
    im = ImageOps.exif_transpose(Image.open(path)).convert('RGB'); a = np.asarray(im.filter(ImageFilter.GaussianBlur(4))).astype(float)
    mean = lambda b: a[b[1]:b[3], b[0]:b[2]].reshape(-1, 3).mean(0); asp = mean(boxes['asphalt'])
    return {p: round(float(np.abs(mean(b)-asp).mean()), 1) for p, b in boxes.items() if p != 'asphalt'}
# Example: contrast('IMG_2089.jpeg', {'asphalt': (2970, 3744, 4000, 4647), 'shirt': (2143, 2066, 2840, 2840), 'jeans': (2220, 2970, 2711, 3873)})

## 13. Reference values, parameters and limitations

In [ ]:
print(json.dumps({k: REF[k] for k in ['main_pixel', 'perk_lost', 'feasibility', 'sigma', 'latency', 'unknown']}, indent=1))
PARAMS = {'fisheye': 'equidistant, 192°, 400×400 px; poses F(0,-2.35,0.65,30°) B(0,2.35,0.95,35°) L(-1,-0.9,1.0,45°) R(1,-0.9,1.0,45°)',
          'top view': '440×440 px over ±7.2 m (3.3 cm/px), flat ground', 'seams': 'β=6°, straight 15–75° step 2.5°, 90°/s',
          'footprint': 'exact projection, ℓmax=10.5 m, 12 samples', 'UA-OSM': f'm0={sm.M0}, λu={sm.LAM_U}, λs={sm.LAM_S}, L2 thr {sm.L2_THR}, L5 thr {sm.L5_THR}',
          'visibility': '≥40 % of unoccluded fisheye pixels visible, debounced over two observations',
          'perception': 'OU error σ=8 cm, τ=0.6 s; latency 100 ms; track loss 0.1/s for 1 s; coasting σ growth 0.03 m/0.1 s, cap 0.5 m',
          'seeds': 'scene i uses seed 1000+i; perception seed 1007+i'}
for k, v in PARAMS.items(): print(f'{k:12s} {v}')

**Limitations (as stated in the paper).** Flat-ground projection, articulated-cylinder people, ideal photometry in the metric; conditions modelled through assumed perception degradation; simulated visibility set; driver warnings fire too often (up to 40 % of frames at night in fog); one-corner scenes with slow reversing only. Real validation requires synchronised four-camera recordings — see the paper's capture plan.

*Citation:* R. Joshi, "UA-OSM: Uncertainty-Aware Object-Shaped Merging to Reduce Object Loss in 360° Surround-View Stitching," 2026. [DOI to be added]